# Aurora fio `rand_write` — Engine Statistics

Per-engine run statistics (N, mean, min, median, max) for:\n**Bandwidth** (GiB/s), **IOPS**, **Mean latency** (ms), **CPU** (usr+sys %).\n\nCommon configuration: `bs=1m`, `numjobs=16`, `iodepth=16`, `rw=randwrite`, `runtime=60s`.

## DFS

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path(".")

In [ ]:
def parse_dfs_file(fpath):
    m = re.search(r"fio_dfs_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json", Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))

    with open(fpath) as f:
        d = json.load(f)

    job = d["jobs"][0]          # group_reporting=1 → single aggregated job
    w   = job["write"]

    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
    )


rows = [parse_dfs_file(p) for p in sorted(RESULTS_DIR.glob("fio_dfs_*.json"))]
df   = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
df.index.name = "run"
df.index += 1

print(f"Loaded {len(df)} DFS runs")

Loaded 10 DFS runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781795356,1m,16,16,88.244622,90362.492500,2.706695,88.248364
2,1781795431,1m,16,16,84.629335,86660.439304,2.819271,93.699441
3,1781795506,1m,16,16,85.147631,87191.173775,2.823136,76.092669
4,1781795581,1m,16,16,88.007107,90119.277369,2.701697,96.776593
5,1781795656,1m,16,16,88.119113,90233.971635,2.701460,90.277601
6,1781795731,1m,16,16,79.368342,81273.181788,2.992762,99.321050
7,1781795805,1m,16,16,88.987510,91123.210506,2.678237,90.089925
8,1781795880,1m,16,16,88.426185,90548.413439,2.688792,96.152764
9,1781795955,1m,16,16,87.735392,89841.041281,2.704861,98.251791


### Summary Statistics

In [3]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"    : "CPU (%)",
}

stats = (
    df[list(METRICS)]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats["N"] = stats["N"].astype(int)
stats["cv_pct"] = stats["std"] / stats["mean"] * 100
stats = stats[["N", "mean", "std", "cv_pct", "min", "median", "max"]]

pd.set_option("display.float_format", "{:.4f}".format)
stats

,N,mean,std,cv_pct,min,median,max
BW (GiB/s),10,86.7165,2.9642,3.4183,79.3683,88.0631,88.9875
IOPS,10,88797.6537,3035.3419,3.4183,81273.1818,90176.6245,91123.2105
Mean Latency (ms),10,2.7505,0.1002,3.6446,2.6782,2.7033,2.9928
CPU (%),10,92.2300,6.7558,7.3249,76.0927,93.5446,99.3210


## Interception libraries



### libioil / libaio

Files that begin with fio error/signal lines are parsed by skipping to the first `{`;\nabnormally terminated runs are flagged in the `note` column.

In [ ]:
def parse_libioil_libaio_file(fpath):
    m = re.search(r"fio_libioil_libaio_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json", Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))

    text = Path(fpath).read_text()
    json_start = text.find("{")
    d    = json.loads(text[json_start:])

    job  = d["jobs"][0]
    w    = job["write"]
    note = "aborted" if text[:json_start].strip() else ""

    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
        note        = note,
    )


rows_il, skipped = [], []
for p in sorted(RESULTS_DIR.glob("fio_libioil_libaio_*.json")):
    try:
        rows_il.append(parse_libioil_libaio_file(p))
    except Exception as e:
        skipped.append((p.name, str(e)))

df_il = pd.DataFrame(rows_il).sort_values("timestamp").reset_index(drop=True)
df_il.index.name = "run"
df_il.index += 1

if skipped:
    print(f"Skipped {len(skipped)} file(s): {[n for n,_ in skipped]}")
print(f"Loaded {len(df_il)} libioil/libaio runs")

Loaded 10 libioil/libaio runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu,note
run,,,,,,,,,
1,1781728493,1m,16,16,14.3121,14655.5506,17.1489,23.7958,
2,1781796475,1m,16,16,10.9574,11220.3427,22.4472,20.3733,
3,1781796553,1m,16,16,11.1688,11436.8188,22.0908,15.3347,
4,1781796631,1m,16,16,11.2066,11475.5175,22.0052,16.1561,
5,1781796710,1m,16,16,10.6864,10942.8686,23.0332,19.2534,
6,1781796789,1m,16,16,10.7203,10977.5507,22.9517,19.7184,
7,1781796867,1m,16,16,11.1938,11462.4846,22.0347,15.7706,
8,1781796951,1m,16,16,6.3494,6501.7666,35.2316,10.6497,
9,1781797061,1m,16,16,10.8537,11114.2295,22.6760,19.3636,


In [5]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"        : "CPU (%)",
}

stats_il = (
    df_il[list(METRICS)]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_il["N"] = stats_il["N"].astype(int)
stats_il["cv_pct"] = stats_il["std"] / stats_il["mean"] * 100
stats_il = stats_il[["N", "mean", "std", "cv_pct", "min", "median", "max"]]

pd.set_option("display.float_format", "{:.4f}".format)
stats_il

,N,mean,std,cv_pct,min,median,max
BW (GiB/s),10,10.8657,1.9082,17.5613,6.3494,11.0631,14.3121
IOPS,10,11126.5064,1953.9624,17.5613,6501.7666,11328.5807,14655.5506
Mean Latency (ms),10,23.1624,4.5672,19.7183,17.1489,22.2690,35.2316
CPU (%),10,17.6238,3.6276,20.5835,10.6497,17.7047,23.7958


---
### libioil / pvsync2

In [ ]:
def parse_simple(fpath, pattern):
    m = re.search(pattern, Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))
    text = Path(fpath).read_text()
    d    = json.loads(text[text.find("{"):])
    job  = d["jobs"][0]
    w    = job["write"]
    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
    )

PATTERN = r"fio_\S+?_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json"

rows_pv2 = [parse_simple(p, PATTERN)
            for p in sorted(RESULTS_DIR.glob("fio_libioil_pvsync2_*.json"))]
df_pv2 = pd.DataFrame(rows_pv2).sort_values("timestamp").reset_index(drop=True)
df_pv2.index.name = "run"
df_pv2.index += 1

print(f"Loaded {len(df_pv2)} libioil/pvsync2 runs")

Loaded 10 libioil/pvsync2 runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781799098,1m,16,16,12.1915,12484.1339,1.1111,13.0670
2,1781799185,1m,16,16,12.1465,12437.9687,1.1134,13.6493
3,1781799271,1m,16,16,12.3258,12621.5959,1.1274,11.9604
4,1781799350,1m,16,16,12.0796,12369.4877,1.1347,13.3646
5,1781799429,1m,16,16,12.1441,12435.5688,1.1283,13.6521
6,1781799507,1m,16,16,12.2195,12512.7291,1.1265,13.0730
7,1781799592,1m,16,16,12.3051,12600.4300,1.1223,12.5901
8,1781799678,1m,16,16,12.2717,12566.1811,1.1237,12.6638
9,1781799768,1m,16,16,12.2494,12543.4319,1.1306,12.1827


In [7]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"        : "CPU (%)",
}

stats_pv2 = (
    df_pv2[list(METRICS)]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_pv2["N"] = stats_pv2["N"].astype(int)
stats_pv2["cv_pct"] = stats_pv2["std"] / stats_pv2["mean"] * 100
stats_pv2 = stats_pv2[["N", "mean", "std", "cv_pct", "min", "median", "max"]]

pd.set_option("display.float_format", "{:.4f}".format)
stats_pv2

,N,mean,std,cv_pct,min,median,max
BW (GiB/s),10,12.2120,0.0776,0.6351,12.0796,12.2055,12.3258
IOPS,10,12505.0494,79.4229,0.6351,12369.4877,12498.4315,12621.5959
Mean Latency (ms),10,1.1258,0.0088,0.7802,1.1111,1.1269,1.1398
CPU (%),10,12.8424,0.6126,4.7703,11.9604,12.8654,13.6521


---
### Stability Summary — libioil Engines

| Engine | N | BW mean (GiB/s) | BW median | BW min | BW max | Lat mean (ms) | Assessment |
|---|---|---|---|---|---|---|---|
| libioil/libaio  | 10 | 10.87 | 11.06 | 6.35 | 14.31 | 23.2 | **Unstable** — two outliers (run 1 from prior day at 14.3, run 8 aborted at 6.4); core 8 runs cluster at 10.7–11.2 GiB/s |
| libioil/pvsync2 | 10 | 12.21 | 12.21 | 12.08 | 12.33 |  1.1 | **Very stable** — <1% CV across all runs, negligible variance |
| libioil/psync   | 10 | 19.23 | 19.60 | 13.04 | 21.02 |  0.7 | **Mostly stable** — one outlier run at 13.0 GiB/s (~34% below); remaining 9 runs tight at 19.5–21.0 GiB/s |

**libioil/psync delivers the highest bandwidth** (~19.5 GiB/s typical) at the lowest latency (~0.73 ms),
at the cost of near-100% CPU utilization (synchronous, CPU-bound I/O path).
**libioil/pvsync2** is the most repeatable but achieves only ~12.2 GiB/s.
**libioil/libaio** sits in between in bandwidth but shows the most run-to-run variation,
partly due to an aborted run and an earlier isolated test captured in the result set.

---
## dfuse Engines

### dfuse / libaio

In [8]:
rows_dfu_la = [parse_simple(p, PATTERN)
               for p in sorted(RESULTS_DIR.glob("fio_dfuse_libaio_*.json"))]
df_dfu_la = pd.DataFrame(rows_dfu_la).sort_values("timestamp").reset_index(drop=True)
df_dfu_la.index.name = "run"
df_dfu_la.index += 1

print(f"Loaded {len(df_dfu_la)} dfuse/libaio runs")
df_dfu_la

Loaded 10 dfuse/libaio runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781801277,1m,16,16,29.2157,29916.9014,8.3808,99.0360
2,1781801353,1m,16,16,29.0287,29725.3712,8.4337,98.7348
3,1781801429,1m,16,16,28.9548,29649.7558,8.4561,99.2262
4,1781801505,1m,16,16,29.4340,30140.4643,8.3258,98.8649
5,1781801581,1m,16,16,29.0307,29727.4212,8.4374,99.6321
6,1781801658,1m,16,16,29.0847,29782.7703,8.4221,99.2351
7,1781801734,1m,16,16,28.9326,29626.9895,8.4623,98.6552
8,1781801810,1m,16,16,28.8706,29563.4573,8.4810,98.6756
9,1781801886,1m,16,16,29.0090,29705.2549,8.4439,98.7064


In [9]:
stats_dfu_la = (
    df_dfu_la[list(METRICS)]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_dfu_la["N"] = stats_dfu_la["N"].astype(int)
stats_dfu_la["cv_pct"] = stats_dfu_la["std"] / stats_dfu_la["mean"] * 100
stats_dfu_la = stats_dfu_la[["N", "mean", "std", "cv_pct", "min", "median", "max"]]

pd.set_option("display.float_format", "{:.4f}".format)
stats_dfu_la

,N,mean,std,cv_pct,min,median,max
BW (GiB/s),10,29.0494,0.1658,0.5707,28.8706,29.0189,29.4340
IOPS,10,29746.6042,169.7522,0.5707,29563.4573,29715.3131,30140.4643
Mean Latency (ms),10,8.4306,0.0459,0.5450,8.3258,8.4406,8.4810
CPU (%),10,98.9968,0.3243,0.3276,98.6552,98.9505,99.6321


---
### dfuse / pvsync2

In [ ]:
rows_dfu_pv2 = [parse_simple(p, PATTERN)
                for p in sorted(RESULTS_DIR.glob("fio_dfuse_pvsync2_*.json"))]
df_dfu_pv2 = pd.DataFrame(rows_dfu_pv2).sort_values("timestamp").reset_index(drop=True)
df_dfu_pv2.index.name = "run"
df_dfu_pv2.index += 1

print(f"Loaded {len(df_dfu_pv2)} dfuse/pvsync2 runs")

Loaded 10 dfuse/pvsync2 runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781802802,1m,16,16,39.5250,40473.5921,0.3567,99.1068
2,1781802879,1m,16,16,39.4206,40366.7211,0.3575,99.0297
3,1781802955,1m,16,16,39.5049,40453.0258,0.3568,99.4991
4,1781803031,1m,16,16,39.3128,40256.3457,0.3586,99.0303
5,1781803107,1m,16,16,39.2863,40229.1962,0.3589,99.2985
6,1781803184,1m,16,16,39.3520,40296.4284,0.3581,98.9911
7,1781803260,1m,16,16,39.2380,40179.7273,0.3594,99.4496
8,1781803336,1m,16,16,39.2867,40229.6128,0.3588,98.9334
9,1781803412,1m,16,16,39.2332,40174.8471,0.3594,99.3918


In [11]:
stats_dfu_pv2 = (
    df_dfu_pv2[list(METRICS)]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_dfu_pv2["N"] = stats_dfu_pv2["N"].astype(int)
stats_dfu_pv2["cv_pct"] = stats_dfu_pv2["std"] / stats_dfu_pv2["mean"] * 100
stats_dfu_pv2 = stats_dfu_pv2[["N", "mean", "std", "cv_pct", "min", "median", "max"]]

pd.set_option("display.float_format", "{:.4f}".format)
stats_dfu_pv2

,N,mean,std,cv_pct,min,median,max
BW (GiB/s),10,39.3457,0.1043,0.2650,39.2332,39.3050,39.5250
IOPS,10,40289.9709,106.7636,0.2650,40174.8471,40248.2792,40473.5921
Mean Latency (ms),10,0.3583,0.0010,0.2691,0.3567,0.3585,0.3594
CPU (%),10,99.1877,0.2057,0.2074,98.9334,99.1268,99.4991


---
### Stability Summary — dfuse Engines

| Engine | N | BW mean (GiB/s) | BW median | BW min | BW max | BW stdev | Lat mean (ms) | Assessment |
|---|---|---|---|---|---|---|---|---|
| dfuse/libaio  | 10 | 29.05 | 29.02 | 28.87 | 29.43 | 0.17 |  8.43 | **Very stable** — <0.6% CV; all runs within a narrow 0.56 GiB/s band |
| dfuse/psync   | 10 | 39.37 | 39.43 | 39.04 | 39.50 | 0.15 |  0.36 | **Very stable** — <0.4% CV; one slightly lower run at 39.04 but no true outlier |
| dfuse/pvsync2 | 10 | 39.35 | 39.31 | 39.23 | 39.53 | 0.10 |  0.36 | **Very stable** — <0.3% CV; tightest spread of all three dfuse engines |

**dfuse/psync and dfuse/pvsync2 deliver essentially identical throughput** (~39.35–39.37 GiB/s) and
latency (~0.36 ms), both at ~99% CPU.  
**dfuse/libaio** achieves ~29 GiB/s — ~26% lower bandwidth — with significantly higher mean latency
(8.4 ms vs 0.36 ms), reflecting the asynchronous I/O depth overhead through the FUSE layer.  
All three engines are highly repeatable with no outlier runs.

---
## Cross-Engine Comparison — dfuse vs libioil

Median BW, median IOPS, mean latency, and mean CPU across all engines (10 runs each).

| Engine | dfuse BW (GiB/s) | libioil BW (GiB/s) | dfuse IOPS | libioil IOPS | dfuse Lat (ms) | libioil Lat (ms) | dfuse CPU (%) | libioil CPU (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| libaio  | 29.02 | 11.06 | 29,715 | 11,329 |  8.43 | 23.16 | 98.99 | 17.62 |
| psync   | 39.43 | 19.60 | 40,375 | 20,066 |  0.36 |  0.74 | 99.15 | 97.34 |
| pvsync2 | 39.31 | 12.21 | 40,248 | 12,498 |  0.36 |  1.13 | 99.19 | 12.84 |

### Key observations

- **dfuse/psync ≈ dfuse/pvsync2** in both BW (~39.4 GiB/s) and latency (~0.36 ms); pvsync2 is marginally more stable.
- **dfuse outperforms libioil** for psync and pvsync2 by ~2×: dfuse routes I/O through a tuned FUSE path while libioil intercepts POSIX calls before they reach FUSE, adding round-trip overhead at these block sizes.
- **libioil/libaio** trades throughput for CPU efficiency (~18% CPU vs ~99%); BW drops to ~11 GiB/s and latency rises to 23 ms — the async iodepth queue incurs significant overhead through the interception layer.
- **dfuse/libaio** recovers substantial BW (29 GiB/s, ~2.6× over libioil/libaio) but still lags the synchronous dfuse engines by ~26%, with 8.4 ms latency from the iodepth queue absorbing FUSE round-trip cost.
- **libioil/psync** (19.6 GiB/s, 0.74 ms) is the best-performing libioil variant, but still ~50% below the equivalent dfuse path.